In [1]:
import time, requests
from typing import Optional, Callable, Dict, Any, List
from enhanced_query_script import wait_for_rate_limit, norm_s2

# Local allowed types aligned with what you request from S2
ALLOWED_S2_TYPES = {"journalarticle", "conference", "review", "preprint"}

def search_semantic_scholar(
    query: str,
    on_record: Optional[Callable[[Dict[str, Any]], None]] = None,
    year_min: int = 2018,
    s2_api_key: Optional[str] = None,
    fields_of_study: Optional[List[str]] = ("Computer Science", "Agricultural and Food Sciences"),
    min_citations: Optional[int] = None,          # e.g., 5 or 10
    open_access_only: bool = False,               # include param if True
    sort: str = "citationCount:desc"
):
    base_url = "https://api.semanticscholar.org/graph/v1/paper/search/bulk"
    headers = {
        "Accept": "application/json",
        "User-Agent": "AgriReviewBot/1.0 (S2 bulk search)",
    }
    if s2_api_key:
        headers["x-api-key"] = s2_api_key

    params = {
        "query": query,
        "year": f"{year_min}-",
        "publicationTypes": "JournalArticle,Conference,Review",
        "sort": sort,
        "fields": "title,abstract,year,authors,venue,publicationTypes,url,externalIds,citationCount,isOpenAccess,openAccessPdf",
    }
    if fields_of_study:
        params["fieldsOfStudy"] = ",".join(fields_of_study)
    if min_citations is not None:
        params["minCitationCount"] = str(min_citations)
    if open_access_only:
        params["openAccessPdf"] = ""  # presence-only flag

    out = []
    session = requests.Session()
    while True:
        wait_for_rate_limit(1.0)
        resp = session.get(base_url, params=params, headers=headers, timeout=30)
        if resp.status_code in (429, 500, 502, 503):
            time.sleep(1.5); continue
        resp.raise_for_status()
        payload = resp.json()

        for item in payload.get("data", []):
            pts = [p.lower() for p in (item.get("publicationTypes") or [])]
            if pts and not any(p in ALLOWED_S2_TYPES for p in pts):
                continue
            rec = norm_s2(item)
            if rec and (not rec["year"] or rec["year"] >= year_min):
                out.append(rec)
                if on_record: on_record(rec)

        # IMPORTANT: use 'token' (not 'next')
        tok = payload.get("token")
        if not tok:
            break
        params["token"] = tok
    return out


In [3]:
import json
from enhanced_query_script import to_jsonl, to_csv
import os, datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = os.path.join("../outputs", f"s2_run_{timestamp}")
os.makedirs(outdir, exist_ok=True)

queries_file = "../queries/queries_s2_boolean.json"
queries = []
with open(queries_file, "r", encoding="utf-8") as f:
    queries = json.load(f)

for idx, q in enumerate(queries, start=1):
    print(f"\n=== Query {idx}/{len(queries)} ===")
    print(q)
    results = search_semantic_scholar(
        q,
        year_min=2018,
        s2_api_key=os.getenv("S2_API_KEY"),                # set your key in env
        fields_of_study=["Computer Science", "Agricultural and Food Sciences"],
        min_citations=None,                                # e.g., 5 to prune
        open_access_only=False
    )
    for r in results:
        r["query"] = q; r["query_id"] = f"Q{idx:02d}"
    to_jsonl(os.path.join(outdir, f"Q{idx:02d}_results.jsonl"), results)
    to_csv  (os.path.join(outdir, f"Q{idx:02d}_results.csv"),   results)
    print(f"Found {len(results)} results")



=== Query 1/16 ===
"field conditions" + ("plant disease" | pest) + (image | imaging | vision | camera | rgb | hyperspectral | uav | drone) + (adapt* | shift | generaliz* | robust*) -human -clinical -patient -medical
Found 34 results

=== Query 2/16 ===
"in field" + (plant | crop) + (disease | pest) + (image | imaging | vision | camera | rgb | hyperspectral | uav | drone) + (adapt* | generaliz* | robust* | shift) -human -clinical
Found 538 results

=== Query 3/16 ===
"lab to field"~2 + ("plant disease" | pest) + (image | vision | camera | uav | drone) + (adapt* | generaliz* | robust* | deployment)
Found 3 results

=== Query 4/16 ===
"domain shift" + ("plant disease" | pest) + (plant | crop) + (image | vision)
Found 5 results

=== Query 5/16 ===
("dataset shift" | "distribution shift") + (plant | crop) + (disease | pest) + (image | vision)
Found 0 results

=== Query 6/16 ===
"domain generalization" + (plant | crop) + (disease | pest) + (image | vision)
Found 5 results

=== Query 7/16 ==

In [5]:
# aggregate all results
import glob
all_results = []
for fn in glob.glob(os.path.join(outdir, "Q??_results.jsonl")):
    with open(fn, "r", encoding="utf-8") as f:
        for line in f:
            all_results.append(json.loads(line))
print(f"Total aggregated results: {len(all_results)}")
to_jsonl(os.path.join(outdir, f"all_results.jsonl"), all_results)
to_csv  (os.path.join(outdir, f"all_results.csv"),   all_results)


Total aggregated results: 615


In [6]:
print(outdir)

../outputs\s2_run_20251113_100016
